In [1]:
from langchain.agents import create_agent
from langchain_tavily import TavilySearch
from langgraph.checkpoint.sqlite import SqliteSaver
from langchain.messages import HumanMessage
import sqlite3
import base64
import urllib.request
from dotenv import load_dotenv

load_dotenv()

web_search = TavilySearch(
    max_results=5,
    topic="general"
)

# langgraph 会在子线程里写入 checkpoint，必须关闭 sqlite 的同线程检查
connection = sqlite3.connect("resource/ai_chef.db", check_same_thread=False)
checkpointer = SqliteSaver(connection)

system_prompt = """
你是一名私人厨师。收到用户提供的食材照片或清单后，请按以下流程操作：
1.识别和评估食材：若用户提供照片，首先辨识所有可见食材。基于食材的外观状态，评估其新鲜度与可用量，整理出一份“当前可用食材清单”。
2.智能食谱检索：优先调用 web_search 工具，以“可用食材清单”为核心关键词，查找可行菜谱。
3.多维度评估与排序：从营养价值和制作难度两个维度对检索到的候选食谱进行量化打分，并根据得分排序，制作简单且营养丰富的排名靠前。
4.结构化方案输出：把排序后的食谱整理为一份结构清晰的建议报告，要包含食谱信息、得分、推荐理由、食谱的参考图片，帮助用户快速做出决策。

请严格按照流程，优先调用 web_search 工具搜索食谱，搜索不到的情况下才能自己发挥。
"""

agent = create_agent(
    model="deepseek-flash",
    tools=[web_search],
    checkpointer=checkpointer,
    system_prompt=system_prompt
)

# 该图片 URL 会 301 跳转到 img.magnific.com，服务端拉取会失败，
# 所以先下载到本地，再用 base64 传给模型
image_url = "https://img.freepik.com/free-photo/arrangement-different-foods-organized-fridge_23-2149099882.jpg"

request = urllib.request.Request(image_url, headers={"User-Agent": "Mozilla/5.0"})
with urllib.request.urlopen(request, timeout=30) as response:
    img_bytes = response.read()
    mime_type = response.headers.get_content_type()

img_base64 = base64.b64encode(img_bytes).decode("utf-8")

multimodal_message = HumanMessage(
    content_blocks=[
        {"type": "image", "base64": img_base64, "mime_type": mime_type},
        {"type": "text", "text": "帮我看看这些食材能做些什么？"}
    ])

config = {"configurable": {"thread_id": "6"}}

response = agent.invoke({"messages": [multimodal_message]}, config)

for msg in response["messages"]:
    msg.pretty_print()


================================ Human Message =================================

[{'type': 'image', 'base64': '...', 'mime_type': 'image/jpeg'}, {'type': 'text', 'text': '帮我看看这些食材能做些什么？'}]
================================== Ai Message ==================================

我先根据照片识别这些食材，再去找可行的菜谱。
Tool Calls:
  tavily_search (call_00_1cp2oqVpSG4IdJ2Ku4m59970)
 Call ID: call_00_1cp2oqVpSG4IdJ2Ku4m59970
  Args:
    query: 三文鱼 蘑菇 西兰花 食谱 做法
    include_images: True
    search_depth: advanced
  tavily_search (call_01_MXPYJBHlrDtLwxOb1pPX1430)
 Call ID: call_01_MXPYJBHlrDtLwxOb1pPX1430
  Args:
    query: 鸡胸肉 彩椒 番茄 食谱 简单做法
    include_images: True
    search_depth: advanced
  tavily_search (call_02_uLoDhpGq6c42Js6pSDe09040)
 Call ID: call_02_uLoDhpGq6c42Js6pSDe09040
  Args:
    query: 口蘑 白蘑菇 小番茄 生菜 沙拉 食谱
    include_images: True
    search_depth: advanced
  tavily_search (call_03_nMBmiM4FdlmijdNpEGUL4304)
 Call ID: call_03_nMBmiM4FdlmijdNpEGUL4304
  Args:
    query: 土豆 洋葱 西兰花 家常菜 食谱
    include_i